In [2]:
import pandas as pd
import plotly.express as px

df=pd.read_csv("/Users/saramolano/clase ciencia de datos/proyecto 2/Proyecto2-G03-DATA1002/data/processed/panel_final.csv")
df["fecha"]=pd.to_datetime(df["mes"] + "-01")
IND=["ocupacion_ema_antioquia", "indice_tarifa_ema_antioquia", "entradas_extranjeros_jmc", "trafico_total"]
print(df.shape, df["fecha"].min().date(), df["fecha"].max().date())
df.isna().sum(),df["mes"].duplicated().sum()

(84, 22) 2019-01-01 2025-12-01


(mes                            0
 anio                           0
 mes_num                        0
 trafico_guarne                 0
 trafico_las_palmas             0
 trafico_total                  0
 trafico_livianos               0
 ocupacion_ema_antioquia        0
 indice_tarifa_ema_sencilla     0
 indice_tarifa_ema_doble        0
 indice_tarifa_ema_antioquia    0
 entradas_extranjeros_jmc       0
 n_eventos                      0
 n_eventos_grandes              0
 n_conciertos                   0
 n_festivales                   0
 n_internacionales              0
 aforo_total                    0
 hay_evento_grande              0
 covid                          0
 temporada_alta                 0
 fecha                          0
 dtype: int64,
 np.int64(0))

## Estadísticas descriptivas
media, mediana, desviación, mínimo y mpaximo de los indicadores

In [2]:
desc=df[IND+["n_eventos_grandes", "aforo_total"]].agg(["mean", "median", "std", "min", "max"]).T.round(1)
desc

,mean,median,std,min,max
ocupacion_ema_antioquia,47.7,50.5,11.9,9.7,63.3
indice_tarifa_ema_antioquia,116.1,113.7,22.5,75.2,153.1
entradas_extranjeros_jmc,50357.5,51474.0,29997.9,5.0,113031.0
trafico_total,986929.6,1045113.0,179427.6,232018.0,1232328.0
n_eventos_grandes,1.1,1.0,1.2,0.0,5.0
aforo_total,14333.1,0.0,25644.4,0.0,112000.0


## Agregaciones por año y por mes

In [3]:
por_anio=df.groupby("anio")[IND+["n_eventos_grandes"]].mean().round(1)
por_mes=df.groupby("mes_num")[IND+["n_eventos_grandes"]].mean().round(1)
display(por_anio, por_mes)

,ocupacion_ema_antioquia,indice_tarifa_ema_antioquia,entradas_extranjeros_jmc,trafico_total,n_eventos_grandes
anio,,,,,
2019,47.8,100.0,32025.1,970933.7,1.4
2020,27.1,86.4,10358.9,666999.5,0.2
2021,42.7,93.9,22343.3,950284.7,0.5
2022,58.9,115.6,51789.4,1071178.2,1.7
2023,54.9,133.0,62703.3,1070613.6,1.3
2024,51.8,137.3,80927.6,1075726.2,1.1
2025,50.7,146.6,92355.1,1102771.2,1.2


,ocupacion_ema_antioquia,indice_tarifa_ema_antioquia,entradas_extranjeros_jmc,trafico_total,n_eventos_grandes
mes_num,,,,,
1,48.9,119.3,48595.3,980066.1,1.6
2,48.3,113.9,48937.0,932863.1,0.6
3,45.4,114.8,45677.7,975097.0,0.3
4,41.4,112.9,42853.0,858264.7,0.7
5,41.6,110.7,43629.3,916983.9,0.4
6,44.2,113.3,46639.1,972375.9,0.3
7,49.4,115.5,58470.1,1008018.3,1.3
8,50.6,118.0,56631.4,1012528.7,1.9
9,47.8,115.0,46772.7,987574.4,0.9


## Variación contra el mismo mes del año anterior (Year over Year)
Los primeros 12 meses quedan vacíos (2019) ya que no hay año anterior

In [4]:
for c in IND:
    df[c+"_yoy%"]=df[c].pct_change(12)*100
df[["mes"]+[c+"_yoy%" for c in IND]].dropna().round(1).tail(12)

,mes,ocupacion_ema_antioquia_yoy%,indice_tarifa_ema_antioquia_yoy%,entradas_extranjeros_jmc_yoy%,trafico_total_yoy%
72,2025-01,-3.7,7.2,36.4,0.4
73,2025-02,-4.6,7.2,19.7,0.8
74,2025-03,-2.7,6.4,18.7,5.8
75,2025-04,0.7,11.7,20.0,3.9
76,2025-05,-1.0,6.2,8.8,1.8
77,2025-06,-2.5,6.4,13.9,2.7
78,2025-07,2.0,7.1,18.2,6.1
79,2025-08,-3.5,9.9,10.5,3.4
80,2025-09,1.0,6.3,6.8,7.8
81,2025-10,1.2,5.2,15.8,1.4


In [11]:
yoy=df.melt(id_vars=["fecha"], value_vars=[c+"_yoy%" for c in IND], var_name="indicador", value_name="yoy_%")
fig=px.line(yoy, x="fecha", y="yoy_%", color="indicador", title="Variación YoY (%)")
fig.add_hline(y=0, line_width=1, line_color="black")
nombres = {
    "ocupacion_ema_antioquia_yoy%": "Ocupación",
    "indice_tarifa_ema_antioquia_yoy%": "Tarifa",
    "entradas_extranjeros_jmc_yoy%": "Extranjeros",
    "trafico_total_yoy%": "Tráfico",
}
yoy2 = yoy[yoy["fecha"] >= "2023-01-01"]   # evita los años base distorsionados por el COVID
yoy2["indicador"] = yoy2["indicador"].map(nombres)
fig = px.line(yoy2, x="fecha", y="yoy_%", facet_row="indicador", color="indicador",
              title="Variación YoY (%) desde 2023", height=750)
fig.update_yaxes(matches=None)             # cada indicador con su propia escala
fig.add_hline(y=0, line_width=1, line_color="black")
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_layout(showlegend=False)
fig.show()


## Meses con y sin eventos grandes
La temporada alta y el COVID confunden la comparación, por eso se compara tambíen sin estos dos

In [6]:
print(df["hay_evento_grande"].value_counts())
sin_covid = df[df["covid"] == 0]
print("Todos los meses:"); display(df.groupby("hay_evento_grande")[IND].mean().round(1))
print("Sin COVID:"); display(sin_covid.groupby("hay_evento_grande")[IND].mean().round(1))
print("Sin COVID, por temporada:"); display(sin_covid.groupby(["temporada_alta", "hay_evento_grande"])[IND].mean().round(1))

hay_evento_grande
1    49
0    35
Name: count, dtype: int64
Todos los meses:


,ocupacion_ema_antioquia,indice_tarifa_ema_antioquia,entradas_extranjeros_jmc,trafico_total
hay_evento_grande,,,,
0,39.3,105.1,34778.7,888276.7
1,53.7,124.0,61485.2,1057395.9


Sin COVID:


,ocupacion_ema_antioquia,indice_tarifa_ema_antioquia,entradas_extranjeros_jmc,trafico_total
hay_evento_grande,,,,
0,48.7,123.7,59188.5,1014807.3
1,54.2,126.5,64490.2,1057876.8


Sin COVID, por temporada:


ocupacion_ema_antioquia  \
temporada_alta hay_evento_grande                            
0              0                                     48.3   
               1                                     53.8   
1              0                                     50.6   
               1                                     54.7   

                                  indice_tarifa_ema_antioquia  \
temporada_alta hay_evento_grande                                
0              0                                        121.0   
               1                                        125.7   
1              0                                        136.6   
               1                                        127.1   

                                  entradas_extranjeros_jmc  trafico_total  
temporada_alta hay_evento_grande                                           
0              0                                   55960.2      1001836.2  
               1                                   61069.1      1034486.9  
1              0                                   74254.0      1075339.0  
               1                                   67762.4      1080249.7

In [7]:
sin_covid.groupby("hay_evento_grande")[IND].mean().pipe(lambda g: (g.loc[1] / g.loc[0] - 1) * 100).round(1).rename("Dif. % (con vs sin evento grande)")

ocupacion_ema_antioquia        11.3
indice_tarifa_ema_antioquia     2.2
entradas_extranjeros_jmc        9.0
trafico_total                   4.2
Name: Dif. % (con vs sin evento grande), dtype: float64

## Correlaciones
Calculadas sin COVID ya que 2020-2021 lo distorsiona

In [8]:
cols = ["n_eventos_grandes", "n_internacionales", "aforo_total"] + IND
corr = sin_covid[cols].corr().round(2)
display(corr.loc[["n_eventos_grandes", "n_internacionales", "aforo_total"], IND])
px.imshow(corr, text_auto=True, color_continuous_scale="RdBu_r", zmin=-1, zmax=1,
          title="Matriz de correlación (sin COVID)").show()

,ocupacion_ema_antioquia,indice_tarifa_ema_antioquia,entradas_extranjeros_jmc,trafico_total
n_eventos_grandes,0.43,0.03,0.05,0.09
n_internacionales,0.38,-0.15,-0.13,-0.01
aforo_total,0.06,0.08,0.12,0.09
